# Phase 1 — HHR benchmark trên DAPR

Notebook này trả lời hai câu hỏi: (1) tổ hợp sparse/dense/combined nào phù hợp ở tầng document và passage; (2) lỗi xảy ra ở document routing hay passage ranking. Logic tái sử dụng nằm trong `src/dapr_hhr`; notebook chỉ cấu hình, chạy và phân tích.

**Protocol:** smoke trước, sau đó tune trên MS MARCO, freeze cấu hình, rồi mới đánh giá zero-shot trên NaturalQuestions, MIRACL, Genomics và ConditionalQA. Không dùng NQ-hard để tune.

In [ ]:
# 1. Bootstrap project (Kaggle hoặc local)
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/ManhTanTran/hierarchical-retrieval-benchmark.git'
REPO_REF = 'main'  # Sau baseline đầu tiên, pin thành commit SHA để tái lập tuyệt đối.
ON_KAGGLE = Path('/kaggle').exists()

if ON_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working/hierarchical-retrieval-benchmark')
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', REPO_REF,
            REPO_URL, str(PROJECT_ROOT)
        ], check=True)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(PROJECT_ROOT / 'requirements-kaggle.txt')
], check=True)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print({'on_kaggle': ON_KAGGLE, 'project_root': str(PROJECT_ROOT)})

## 2. Cấu hình tập trung

`smoke` chạy bốn cấu hình đại diện. `full` đăng ký đủ chín cấu hình, nhưng hãy chạy từng dataset và dùng indexed backend cho MIRACL/Genomics đầy đủ.

In [ ]:
from dataclasses import replace

from dapr_hhr.config import load_config

RUN_MODE = 'smoke'              # 'smoke' hoặc 'full'
DATASET_NAME = 'ConditionalQA'  # MSMARCO, NaturalQuestions, MIRACL, Genomics, ConditionalQA
QUERY_LIMIT = 25 if RUN_MODE == 'smoke' else None
CORPUS_LIMIT = 5_000 if RUN_MODE == 'smoke' else None
FUSION = 'rrf'                  # ablation: 'interleave'

CACHE_DIR = Path('/kaggle/working/dapr_hhr_cache') if ON_KAGGLE else PROJECT_ROOT / 'cache'
OUTPUT_DIR = Path('/kaggle/working/dapr_hhr_outputs') if ON_KAGGLE else PROJECT_ROOT / 'outputs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = load_config(PROJECT_ROOT / 'configs' / 'phase1.yaml')
config = replace(
    config,
    data=replace(
        config.data, dataset_name=DATASET_NAME,
        query_limit=QUERY_LIMIT, corpus_limit=CORPUS_LIMIT
    ),
    retrieval=replace(config.retrieval, fusion=FUSION),
    run=replace(config.run, mode=RUN_MODE),
)
config

## 3. Load và validate DAPR

Ở smoke mode, adapter giữ các passage đầu tiên và tự bổ sung toàn bộ gold passage cho những query đã chọn. Nhờ vậy Recall/nDCG vẫn có ground truth hợp lệ.

In [ ]:
from dapr_hhr.data import DAPRDatasetAdapter, summarize_dataset

adapter = DAPRDatasetAdapter(config.data.hf_repo)
bundle = adapter.load_bundle(
    dataset_name=config.data.dataset_name,
    split=config.data.split,
    query_limit=config.data.query_limit,
    corpus_limit=config.data.corpus_limit,
    preserve_gold=config.data.preserve_gold,
)
summary = summarize_dataset(bundle)
assert summary['queries_with_relevant_passages'] > 0, 'Sample không còn qrel dương.'
summary

## 4. Chọn experiment registry

Tên experiment là `document_method__passage_method`. Full mode dùng đủ 9 tổ hợp.

In [ ]:
from dapr_hhr.experiments import build_experiment_registry

registry = build_experiment_registry()
experiment_names = (
    list(config.run.smoke_experiments)
    if RUN_MODE == 'smoke'
    else list(registry)
)
assert all(name in registry for name in experiment_names)
experiment_names

## 5. Chạy HHR

Dense embedding được cache trong `/kaggle/working`, nên những experiment sau không encode lại cùng collection. Metric dưới đây là kết quả thật của run hiện tại; notebook không chứa số dựng sẵn.

In [ ]:
import pandas as pd

from dapr_hhr.artifacts import save_run_artifacts
from dapr_hhr.experiments import run_hhr_experiment

results = {}
rows = []
for name in experiment_names:
    print(f'\n=== {name} ===')
    result = run_hhr_experiment(
        registry[name], bundle, config, cache_dir=CACHE_DIR
    )
    results[name] = result
    run_dir = save_run_artifacts(OUTPUT_DIR, name, result)
    rows.append({'experiment': name, **result['metrics'], 'artifact_dir': str(run_dir)})

leaderboard = pd.DataFrame(rows).sort_values(
    f'passage_ndcg@{config.evaluation.ndcg_k}', ascending=False
).reset_index(drop=True)
leaderboard

## 6. Diagnose document routing vs passage ranking

- Document recall thấp: query chưa route được tới đúng document; ưu tiên thay document retriever/context.
- Document recall cao nhưng passage recall thấp: passage stage là bottleneck; khi đó mới đáng thử cross-encoder kiểu HiREC.

In [ ]:
metric_columns = [
    'experiment',
    f'document_recall@{config.retrieval.document_top_k}',
    f'passage_recall@{config.evaluation.recall_k}',
    f'document_ndcg@{config.evaluation.ndcg_k}',
    f'passage_ndcg@{config.evaluation.ndcg_k}',
    'latency_ms',
]
leaderboard[metric_columns]

## 7. NQ-hard category analysis (chỉ NaturalQuestions)

Các nhóm CR, MT, MHR, AC chỉ tồn tại ở NQ-hard. Cell này không gán các nhãn đó cho dataset khác.

In [ ]:
from dapr_hhr.metrics import evaluate_by_group

if DATASET_NAME == 'NaturalQuestions':
    query_metadata = adapter.load_query_metadata(DATASET_NAME)
    best_name = leaderboard.iloc[0]['experiment']
    grouped = evaluate_by_group(results[best_name]['per_query'], query_metadata)
    display(pd.DataFrame(grouped).T)
else:
    print('Bỏ qua: category CR/MT/MHR/AC chỉ có trong NQ-hard.')

## 8. Export

Tải file ZIP từ Kaggle Output. Giữ lại cache embedding dưới dạng private Kaggle Dataset nếu muốn tái sử dụng ở session sau.

In [ ]:
import shutil

archive_base = (
    Path('/kaggle/working/dapr_hhr_outputs')
    if ON_KAGGLE
    else PROJECT_ROOT / 'dapr_hhr_outputs'
)
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR)
print(f'Artifacts: {OUTPUT_DIR}')
print(f'Archive:   {archive_path}')